# Online Retail II — Data Exploration (Day 1)

**Dataset:** Online Retail II from UCI Machine Learning Repository  
**Source:** https://archive.ics.uci.edu/dataset/502/online+retail+ii  
**Description:** Transactions from a UK-based online retailer, Dec 2009 – Dec 2011.

## Goal of this notebook

Just look at the data. Understand what columns exist, what the data types are, how big it is, and where the mess lives. No cleaning yet — I want to see it raw first.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max.columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)

In [4]:
# The xlsx has 2 sheets; load both and stack them.
path = "/Users/edi/Documents/online-retail-analysis/data/online_retail_II.xlsx"

df1 = pd.read_excel(path, sheet_name = "Year 2009-2010")
df2 = pd.read_excel(path, sheet_name = "Year 2010-2011")

print(f"Sheet 1 (2009-2010): {df1.shape[0]:,} rows, {df1.shape[1]} columns")
print(f"Sheet 2 (2010-2011): {df2.shape[0]:,} rows, {df2.shape[1]} columns")

df = pd.concat([df1, df2], ignore_index=True)
print(f"\nCombined: {df.shape[0]:,} rows, {df.shape[1]} columns")


Sheet 1 (2009-2010): 525,461 rows, 8 columns
Sheet 2 (2010-2011): 541,910 rows, 8 columns

Combined: 1,067,371 rows, 8 columns


In [5]:
# What do the columns look like?
df.head(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,"13,085.00",United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,"13,085.00",United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,"13,085.00",United Kingdom
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,"13,085.00",United Kingdom
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,"13,085.00",United Kingdom
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,"13,085.00",United Kingdom
8,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,"13,085.00",United Kingdom
9,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01 07:46:00,3.75,"13,085.00",United Kingdom


In [6]:
# Column types and null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 65.1+ MB


In [8]:
# Null counts in nicer format

null_summary = pd.DataFrame({
    "null_count": df.isnull().sum(),
    "null_pct": (df.isnull().sum()/len(df)*100).round(2),
    "dtype": df.dtypes
})

null_summary

,null_count,null_pct,dtype
Invoice,0,0.00,object
StockCode,0,0.00,object
Description,4382,0.41,object
Quantity,0,0.00,int64
InvoiceDate,0,0.00,datetime64[ns]
Price,0,0.00,float64
Customer ID,243007,22.77,float64
Country,0,0.00,object


## Initial Observations of the Data

Refer to the null_summary above. Note which columns have nulls and roughly what percentage. This tells us what needs attention during cleaning.

In [10]:
# Basic Numeric summary of Quantity and Price

df[["Quantity", "Price"]].describe()

,Quantity,Price
count,"1,067,371.00","1,067,371.00"
mean,9.94,4.65
std,172.71,123.55
min,"-80,995.00","-53,594.36"
25%,1.00,1.25
50%,3.00,2.10
75%,10.00,4.15
max,"80,995.00","38,970.00"


Notice that there is negative quantity and price as well as extremely large and small values

In [11]:
# How many unique values in key columns?

for col in ["Invoice", "StockCode", "Description", "Customer ID", "Country"]:
    print(f"{col:<15} {df[col].nunique():>10,} unique")

Invoice             53,628 unique
StockCode            5,305 unique
Description          5,698 unique
Customer ID          5,942 unique
Country                 43 unique


In [14]:
# Date range
print(f"Earliest date: {df['InvoiceDate'].min()}")
print(f"Latest date:   {df['InvoiceDate'].max()}")

Earliest date: 2009-12-01 07:45:00
Latest date:   2011-12-09 12:50:00


In [15]:
# Countries - most e-commerce retail datasets are dominated by one country.
df["Country"].value_counts().head(15)

Country
United Kingdom     981330
EIRE                17866
Germany             17624
France              14330
Netherlands          5140
Spain                3811
Switzerland          3189
Belgium              3123
Portugal             2620
Australia            1913
Channel Islands      1664
Italy                1534
Norway               1455
Sweden               1364
Cyprus               1176
Name: count, dtype: int64

In [16]:
# Some of the invoice values start with a 'C' which is weird. 
# The real invoice numbers are 6 digits. The ones that were cancelled start with 'C'
df["Invoice"] = df["Invoice"].astype(str)
df["Invoice_prefix"] = df["Invoice"].str[0]
df["Invoice_prefix"].value_counts()

Invoice_prefix
5    939382
4    108489
C     19494
A         6
Name: count, dtype: int64

In [18]:
# Analyze the cancelled transcations

cancellations = df[df["Invoice"].str.startswith("C")]
print(f"Cancelled transactions: {len(cancellations):,} rows ({len(cancellations)/len(df)*100:.2f}% of data)")
print("\nSample of cancellations:")
cancellations.head()

Cancelled transactions: 19,494 rows (1.83% of data)

Sample of cancellations:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Invoice_prefix
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,"16,321.00",Australia,C
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,"16,321.00",Australia,C
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,"16,321.00",Australia,C
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,"16,321.00",Australia,C
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,"16,321.00",Australia,C


In [30]:
# Inspect the 'A' prefix invoices
a_invoices = df[df["Invoice"].str.startswith("A")]
print(f"'A' prefix rows: {len(a_invoices)}")
a_invoices

'A' prefix rows: 6


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Invoice_prefix
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,"-53,594.36",NaN,United Kingdom,A
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,"-44,031.79",NaN,United Kingdom,A
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,"-38,925.87",NaN,United Kingdom,A
825443,A563185,B,Adjust bad debt,1,2011-08-12 14:50:00,"11,062.06",NaN,United Kingdom,A
825444,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,"-11,062.06",NaN,United Kingdom,A
825445,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,"-11,062.06",NaN,United Kingdom,A


### Invoice prefix inspection

The `Invoice` column is mostly pure 6-digit numbers, but some prefixes exist:
- **No prefix** (~99%+): normal customer sales
- **'C' prefix**: cancellations (quantities negative)
- **'A' prefix** (6 rows): accounting adjustments — specifically "Adjust bad debt" 
  entries with no Customer ID, using Price to record the dollar value of the 
  write-off. These are not customer transactions and will be excluded from 
  downstream analysis.

Notable: three 'A' rows on 2011-08-12 posted one minute apart with values 
+$11,062.06, -$11,062.06, -$11,062.06 — an accounting correction being 
posted and reversed.

In [19]:
# StockCode is similarly messy — inspect the ones that aren't purely numeric
unusual_codes = df[~df["StockCode"].astype(str).str.match(r"^\d+[A-Z]?$", na=False)]
print(f"Unusual stock codes: {unusual_codes['StockCode'].nunique()} unique codes")
print("\nSample of unusual codes and their descriptions:")
unusual_codes[["StockCode", "Description"]].drop_duplicates().head(20)

Unusual stock codes: 240 unique codes

Sample of unusual codes and their descriptions:


,StockCode,Description
89,POST,POSTAGE
572,79323LP,LIGHT PINK CHERRY LIGHTS
613,15056BL,EDWARDIAN PARASOL BLACK
735,D,Discount
1762,79323GR,GREEN CHERRY LIGHTS
2377,DCGS0058,MISO PRETTY GUM
2378,DCGS0068,DOGS NIGHT COLLAR
2379,DOT,DOTCOM POSTAGE
2533,72349b,SET/6 PURPLE BUTTERFLY T-LIGHTS
2534,72529w,WHITE ADVENT CANDLE


In [28]:
# Weird free text in Description — look for all-caps manual entries
# or descriptions that look like adjustments rather than products
print("Sample descriptions (random 20):")
df["Description"].dropna().sample(20, random_state=42).values

Sample descriptions (random 20):


array(['CHARLOTTE BAG SUKI DESIGN', 'ANGEL DECORATION WITH LACE PADDED',
       'JARDIN ETCHED GLASS FRUITBOWL', 'JUMBO BAG PINK WITH WHITE SPOTS',
       "POPPY'S PLAYHOUSE KITCHEN", 'ASSORTED COLOURS SILK FAN',
       '72 SWEETHEART FAIRY CAKE CASES', 'IVY HEART WREATH',
       'BAKING SET 9 PIECE RETROSPOT ',
       'ASSTD FRUIT+FLOWERS FRIDGE MAGNETS',
       'GIRLS VINTAGE TIN SEASIDE BUCKET',
       'CHRISTMAS LIGHTS 10 VINTAGE BAUBLES',
       'CHARLIE+LOLA RED HOT WATER BOTTLE ', 'PAPER BUNTING RETRO SPOTS',
       'RED WOOLLY HOTTIE WHITE HEART.', 'RECYCLING BAG RETROSPOT ',
       'FOLKART ZINC HEART CHRISTMAS DEC', 'VINTAGE UNION JACK BUNTING',
       'CARD CIRCUS PARADE', 'RED RETROSPOT ROUND CAKE TINS'],
      dtype=object)

## What I found today

**Data shape:** roughly 1M+ rows combined.

**Messy things to handle on Day 2:**
- [ ] `Customer ID` has many nulls — decide how to handle
- [ ] Some `Description` values are missing
- [ ] `Quantity` has negative values (likely cancellations)
- [ ] `Price` has zeros and possibly negatives
- [ ] `Invoice` numbers starting with "C" are cancellations
- [ ] 'A' prefix invoices (6 rows) = "Adjust bad debt" accounting entries.
      Not customer transactions. Exclude from revenue/customer/product analysis.
- [ ] Several unusual `StockCode` values that aren't real products (POST, M, DOT, BANK CHARGES, etc.)
- [ ] Descriptions with suspicious free-text ("damaged", "manual", etc.)
- [ ] Possible duplicate rows

**Data is valid for:**
- Revenue analysis once Price × Quantity is computed and cleaned
- Customer segmentation for rows where Customer ID is populated
- Time-series trends (InvoiceDate looks clean)
- Country-level analysis (UK dominant, but 40+ countries represented)